# ML v1.4 - уточняющая проверка LightGBM

Цель ноутбука: подтвердить LightGBM после победы в `v1.3`, проверить несколько конфигураций вокруг `lightgbm_default`, добавить ROC/FPR-операционные метрики (`Recall@FPR`) и сохранить результаты для сравнения. Датасет и feature policy не меняются: это проверка модели, а не новый feature engineering.

In [ ]:
from pathlib import Path
import sys
import subprocess
import importlib.util
import warnings

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Google Drive mount skipped:", exc)

BASE_PATH = Path("/content/drive/MyDrive/ieee-db")
DATA_FILE = BASE_PATH / "client_profile_v1_0_shuffled.csv"
RESULTS_DIR = BASE_PATH / "ML" / "1.4" / "results_lightgbm_confirm"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Файл данных:", DATA_FILE)
print("Папка результатов:", RESULTS_DIR)

In [ ]:
if importlib.util.find_spec("lightgbm") is None:
    print("lightgbm не найден. Устанавливаю...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
else:
    print("lightgbm уже установлен.")

if importlib.util.find_spec("joblib") is None:
    print("joblib не найден. Устанавливаю...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "joblib"])
else:
    print("joblib уже установлен.")

import time
import numpy as np
import pandas as pd
import joblib

import lightgbm as lgb
from lightgbm import LGBMClassifier

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)

## Настройки запуска

`PREFER_GPU=True` означает: сначала пробуем LightGBM GPU. Если Colab/LightGBM сборка не поддерживает GPU, notebook автоматически повторит обучение этой же конфигурации на CPU.

`SEEDS` можно сократить до `[42]`, если нужно быстро проверить notebook, или расширить, если хотите более надежное подтверждение.

In [ ]:
PREFER_GPU = True
GPU_FALLBACK_TO_CPU = True

SEEDS = [42, 2026, 777]
HIGH_MISSING_THRESHOLD = 0.95

THRESHOLDS = [0.50, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
TOPK_RATES = [0.01, 0.03, 0.05, 0.10, 0.15]
FPR_TARGETS = [0.005, 0.01, 0.02, 0.03, 0.05]

print("PREFER_GPU:", PREFER_GPU)
print("SEEDS:", SEEDS)

In [ ]:
df = pd.read_csv(DATA_FILE)
print("Размер таблицы:", df.shape)

TARGET_COL = "profile_fraud_label"
ID_COLS = ["client_id"]
LEAKY_COLS = ["tx_count_fraud", "fraud_rate"]

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column not found: {TARGET_COL}")

print("Распределение target:")
print(df[TARGET_COL].value_counts(dropna=False))
print("Доля fraud:", float(df[TARGET_COL].mean()))

y = df[TARGET_COL].astype(int)

missing_rate = df.drop(columns=[TARGET_COL], errors="ignore").isna().mean()
high_missing_cols = missing_rate[missing_rate > HIGH_MISSING_THRESHOLD].index.tolist()

exclude_cols = [c for c in ID_COLS + LEAKY_COLS + high_missing_cols if c in df.columns]
features = [c for c in df.columns if c not in exclude_cols + [TARGET_COL]]
X = df[features].copy()

numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print("Исключено колонок:", len(exclude_cols))
print(exclude_cols)
print("Всего признаков:", len(features))
print("Числовые:", len(numeric_cols))
print("Категориальные:", len(categorical_cols))
print("Категориальные признаки:", categorical_cols)

In [ ]:
# Используем тот же принцип split, что и в предыдущих экспериментах: train / val / test.
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
val_size_from_trainval = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=val_size_from_trainval, random_state=42, stratify=y_trainval
)

print("Train:", X_train.shape, "доля fraud:", float(y_train.mean()))
print("Val:", X_val.shape, "доля fraud:", float(y_val.mean()))
print("Test:", X_test.shape, "доля fraud:", float(y_test.mean()))

In [ ]:
# Для LightGBM числовые пропуски заполняются медианой, категориальные - значением MISSING.
try:
    ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse=True)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
            ("onehot", ohe),
        ]), categorical_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

print("Обучаю preprocessing...")
X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc = preprocessor.transform(X_val)
X_test_enc = preprocessor.transform(X_test)

try:
    encoded_feature_names = preprocessor.get_feature_names_out()
except Exception:
    encoded_feature_names = np.array([f"f_{i}" for i in range(X_train_enc.shape[1])])

print("Размер encoded train:", X_train_enc.shape)
print("Размер encoded val:", X_val_enc.shape)
print("Размер encoded test:", X_test_enc.shape)

In [ ]:
def safe_div(num, den):
    return float(num) / float(den) if den else 0.0


def calc_metrics(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    pred = (y_prob >= 0.5).astype(int)
    out = {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "precision_0_5": precision_score(y_true, pred, zero_division=0),
        "recall_0_5": recall_score(y_true, pred, zero_division=0),
        "f1_0_5": f1_score(y_true, pred, zero_division=0),
        "pred_positive_rate": float(pred.mean()),
    }
    # Partial AUC от sklearn используется как индикатор качества в зоне низкого FPR.
    for max_fpr in [0.01, 0.03, 0.05, 0.10]:
        try:
            out[f"partial_roc_auc_fpr_{str(max_fpr).replace('.', '_')}"] = roc_auc_score(y_true, y_prob, max_fpr=max_fpr)
        except Exception:
            out[f"partial_roc_auc_fpr_{str(max_fpr).replace('.', '_')}"] = np.nan
    return out


def threshold_table(model_name, split_name, y_true, y_prob):
    rows = []
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    for thr in THRESHOLDS:
        pred = (y_prob >= thr).astype(int)
        rows.append({
            "model": model_name,
            "split": split_name,
            "threshold": thr,
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "pred_positive_rate": float(pred.mean()),
            "pred_positive_count": int(pred.sum()),
        })
    return pd.DataFrame(rows)


def topk_table(model_name, split_name, y_true, y_prob):
    rows = []
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    order = np.argsort(-y_prob)
    total_pos = int(y_true.sum())
    n = len(y_true)
    for rate in TOPK_RATES:
        k = max(1, int(np.ceil(n * rate)))
        idx = order[:k]
        found = int(y_true[idx].sum())
        rows.append({
            "model": model_name,
            "split": split_name,
            "top_rate": rate,
            "k": k,
            "fraud_found": found,
            "total_fraud": total_pos,
            "precision_at_k": safe_div(found, k),
            "recall_at_k": safe_div(found, total_pos),
        })
    return pd.DataFrame(rows)


def roc_operating_points(model_name, split_name, y_true, y_prob):
    rows = []
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    for target_fpr in FPR_TARGETS:
        valid = np.where(fpr <= target_fpr)[0]
        if len(valid) == 0:
            idx = int(np.argmin(fpr))
        else:
            # Берем точку с максимальным recall при заданном ограничении FPR.
            idx = int(valid[np.argmax(tpr[valid])])
        thr = float(thresholds[idx])
        pred = (y_prob >= thr).astype(int)
        rows.append({
            "model": model_name,
            "split": split_name,
            "target_fpr": target_fpr,
            "actual_fpr": float(fpr[idx]),
            "recall_at_fpr": float(tpr[idx]),
            "threshold": thr,
            "precision_at_fpr": precision_score(y_true, pred, zero_division=0),
            "pred_positive_rate": float(pred.mean()),
            "pred_positive_count": int(pred.sum()),
        })
    return pd.DataFrame(rows)

In [ ]:
base_params = dict(
    objective="binary",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=80,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=2.0,
    class_weight="balanced",
    n_jobs=-1,
    verbose=-1,
)

regularized_params = dict(
    objective="binary",
    n_estimators=3500,
    learning_rate=0.025,
    num_leaves=31,
    min_child_samples=120,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.5,
    reg_lambda=4.0,
    class_weight="balanced",
    n_jobs=-1,
    verbose=-1,
)

more_leaves_params = dict(
    objective="binary",
    n_estimators=3500,
    learning_rate=0.025,
    num_leaves=95,
    min_child_samples=100,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=3.0,
    class_weight="balanced",
    n_jobs=-1,
    verbose=-1,
)

unweighted_auc_params = dict(
    objective="binary",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=80,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=2.0,
    class_weight=None,
    n_jobs=-1,
    verbose=-1,
)

CONFIGS = [
    {"name": "lgbm_pr_default", "params": base_params, "eval_metric": "average_precision", "early_stopping_rounds": 150},
    {"name": "lgbm_auc_default", "params": base_params, "eval_metric": "auc", "early_stopping_rounds": 150},
    {"name": "lgbm_auc_more_leaves", "params": more_leaves_params, "eval_metric": "auc", "early_stopping_rounds": 150},
    {"name": "lgbm_pr_regularized", "params": regularized_params, "eval_metric": "average_precision", "early_stopping_rounds": 150},
    {"name": "lgbm_auc_unweighted", "params": unweighted_auc_params, "eval_metric": "auc", "early_stopping_rounds": 150},
]

print("Конфигурации:")
for cfg in CONFIGS:
    print("-", cfg["name"], "metric=", cfg["eval_metric"])

In [ ]:
model_results = []
threshold_rows = []
topk_rows = []
roc_operating_rows = []
feature_importance_rows = []
fitted_models = {}
prob_cache = {}


def save_intermediate():
    pd.DataFrame(model_results).to_csv(RESULTS_DIR / "ml_v1_4_lightgbm_results.csv", index=False)
    if threshold_rows:
        pd.concat(threshold_rows, ignore_index=True).to_csv(RESULTS_DIR / "ml_v1_4_thresholds.csv", index=False)
    if topk_rows:
        pd.concat(topk_rows, ignore_index=True).to_csv(RESULTS_DIR / "ml_v1_4_topk.csv", index=False)
    if roc_operating_rows:
        pd.concat(roc_operating_rows, ignore_index=True).to_csv(RESULTS_DIR / "ml_v1_4_roc_operating_points.csv", index=False)
    if feature_importance_rows:
        pd.concat(feature_importance_rows, ignore_index=True).to_csv(RESULTS_DIR / "ml_v1_4_feature_importance.csv", index=False)


def fit_one_lgbm(model_name, params, eval_metric, seed, prefer_gpu=True, early_stopping_rounds=150):
    fit_params = dict(params)
    fit_params["random_state"] = seed
    fit_params["feature_fraction_seed"] = seed
    fit_params["bagging_seed"] = seed
    fit_params["data_random_seed"] = seed
    if Использовать GPU:
        fit_params["device_type"] = "gpu"
        fit_params["gpu_use_dp"] = False
    else:
        fit_params["device_type"] = "cpu"

    model = LGBMClassifier(**fit_params)
    start = time.time()
    try:
        model.fit(
            X_train_enc,
            y_train,
            eval_set=[(X_val_enc, y_val)],
            eval_metric=eval_metric,
            callbacks=[lgb.early_stopping(early_stopping_rounds), lgb.log_evaluation(100)],
        )
        elapsed = time.time() - start
        used_device = fit_params["device_type"]
        return model, elapsed, used_device, None
    except Exception as exc:
        if prefer_gpu and GPU_FALLBACK_TO_CPU:
            print("GPU-обучение не удалось. Повторяю на CPU. Ошибка:", repr(exc))
            fit_params["device_type"] = "cpu"
            fit_params.pop("gpu_use_dp", None)
            model = LGBMClassifier(**fit_params)
            start = time.time()
            model.fit(
                X_train_enc,
                y_train,
                eval_set=[(X_val_enc, y_val)],
                eval_metric=eval_metric,
                callbacks=[lgb.early_stopping(early_stopping_rounds), lgb.log_evaluation(100)],
            )
            elapsed = time.time() - start
            return model, elapsed, "cpu_after_gpu_fallback", repr(exc)
        raise


for cfg in CONFIGS:
    for seed in SEEDS:
        model_name = f"{cfg['name']}_seed{seed}"
        print("\n" + "=" * 110)
        print("Обучаю", model_name)
        print("Метрика обучения:", cfg["eval_metric"], "Использовать GPU:", PREFER_GPU)

        model, elapsed, used_device, fallback_error = fit_one_lgbm(
            model_name=model_name,
            params=cfg["params"],
            eval_metric=cfg["eval_metric"],
            seed=seed,
            prefer_gpu=PREFER_GPU,
            early_stopping_rounds=cfg.get("early_stopping_rounds", 150),
        )
        fitted_models[model_name] = model

        best_iter = getattr(model, "best_iteration_", None)
        print("Готово:", model_name, "время, сек:", round(elapsed, 2), "устройство:", used_device, "лучшая итерация:", best_iter)

        for split_name, X_part, y_part in [("val", X_val_enc, y_val), ("test", X_test_enc, y_test)]:
            prob = model.predict_proba(X_part)[:, 1]
            prob_cache[(model_name, split_name)] = prob
            row = calc_metrics(y_part, prob)
            row.update({
                "model": model_name,
                "base_config": cfg["name"],
                "split": split_name,
                "seed": seed,
                "eval_metric": cfg["eval_metric"],
                "elapsed_sec": elapsed,
                "used_device": used_device,
                "best_iteration": best_iter,
                "gpu_fallback_error": fallback_error,
            })
            model_results.append(row)
            threshold_rows.append(threshold_table(model_name, split_name, y_part, prob))
            topk_rows.append(topk_table(model_name, split_name, y_part, prob))
            roc_operating_rows.append(roc_operating_points(model_name, split_name, y_part, prob))

        fi = pd.DataFrame({
            "feature": encoded_feature_names,
            "importance": model.feature_importances_,
            "model": model_name,
            "base_config": cfg["name"],
            "seed": seed,
        }).sort_values("importance", ascending=False).head(250)
        feature_importance_rows.append(fi)

        save_intermediate()
        print("Промежуточные результаты сохранены в:", RESULTS_DIR)

In [ ]:
results_df = pd.DataFrame(model_results)
thresholds_df = pd.concat(threshold_rows, ignore_index=True) if threshold_rows else pd.DataFrame()
topk_df = pd.concat(topk_rows, ignore_index=True) if topk_rows else pd.DataFrame()
roc_ops_df = pd.concat(roc_operating_rows, ignore_index=True) if roc_operating_rows else pd.DataFrame()
importance_df = pd.concat(feature_importance_rows, ignore_index=True) if feature_importance_rows else pd.DataFrame()

results_df.to_csv(RESULTS_DIR / "ml_v1_4_lightgbm_results.csv", index=False)
thresholds_df.to_csv(RESULTS_DIR / "ml_v1_4_thresholds.csv", index=False)
topk_df.to_csv(RESULTS_DIR / "ml_v1_4_topk.csv", index=False)
roc_ops_df.to_csv(RESULTS_DIR / "ml_v1_4_roc_operating_points.csv", index=False)
importance_df.to_csv(RESULTS_DIR / "ml_v1_4_feature_importance.csv", index=False)

сводка_by_config = (
    results_df[results_df["split"] == "test"]
    .groupby("base_config")
    .agg(
        n_runs=("model", "count"),
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
        pr_auc_max=("pr_auc", "max"),
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        roc_auc_max=("roc_auc", "max"),
        partial_auc_5_mean=("partial_roc_auc_fpr_0_05", "mean"),
        precision_0_5_mean=("precision_0_5", "mean"),
        recall_0_5_mean=("recall_0_5", "mean"),
        elapsed_sec_mean=("elapsed_sec", "mean"),
    )
    .reset_index()
    .sort_values(["pr_auc_mean", "roc_auc_mean"], ascending=False)
)
сводка_by_config.to_csv(RESULTS_DIR / "ml_v1_4_сводка_by_config.csv", index=False)

print("Все результаты сохранены в:", RESULTS_DIR)
print("Результаты test, сортировка по PR-AUC:")
display(results_df[results_df["split"] == "test"].sort_values("pr_auc", ascending=False))

print("Результаты test, сортировка по ROC-AUC:")
display(results_df[results_df["split"] == "test"].sort_values("roc_auc", ascending=False))

print("Сводка по конфигурациям:")
display(сводка_by_config)

print("Top-k на test, сортировка по доле top и precision:")
display(topk_df[topk_df["split"] == "test"].sort_values(["top_rate", "precision_at_k"], ascending=[True, False]).head(60))

print("Рабочие точки ROC на test:")
display(roc_ops_df[roc_ops_df["split"] == "test"].sort_values(["target_fpr", "recall_at_fpr"], ascending=[True, False]).head(80))

In [ ]:
# Сохраняем лучшие модели: отдельно по val PR-AUC и отдельно по val ROC-AUC.
val_rows = results_df[results_df["split"] == "val"].copy()

best_by_pr_name = val_rows.sort_values("pr_auc", ascending=False).iloc[0]["model"]
best_by_roc_name = val_rows.sort_values("roc_auc", ascending=False).iloc[0]["model"]

print("Лучшая модель по val PR-AUC:", best_by_pr_name)
print("Лучшая модель по val ROC-AUC:", best_by_roc_name)

best_by_pr_bundle = {
    "model_name": best_by_pr_name,
    "model": fitted_models[best_by_pr_name],
    "preprocessor": preprocessor,
    "features": features,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "selection_rule": {
        "target_col": TARGET_COL,
        "excluded_cols": exclude_cols,
        "high_missing_threshold": HIGH_MISSING_THRESHOLD,
    },
}

best_by_roc_bundle = {
    "model_name": best_by_roc_name,
    "model": fitted_models[best_by_roc_name],
    "preprocessor": preprocessor,
    "features": features,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "selection_rule": {
        "target_col": TARGET_COL,
        "excluded_cols": exclude_cols,
        "high_missing_threshold": HIGH_MISSING_THRESHOLD,
    },
}

joblib.dump(best_by_pr_bundle, RESULTS_DIR / "ml_v1_4_best_by_pr_auc.joblib")
joblib.dump(best_by_roc_bundle, RESULTS_DIR / "ml_v1_4_best_by_roc_auc.joblib")

print("Лучшие модели сохранены:")
print(RESULTS_DIR / "ml_v1_4_best_by_pr_auc.joblib")
print(RESULTS_DIR / "ml_v1_4_best_by_roc_auc.joblib")

In [ ]:
# Экспорт top-5000 клиентов по fraud risk-score.
# Сохраняем два списка: по лучшей модели на val PR-AUC и по лучшей модели на val ROC-AUC.
# Если это одна и та же модель, файлы просто будут почти одинаковыми по логике выбора модели.

split_map = pd.Series("unknown", index=df.index, dtype="object")
split_map.loc[X_train.index] = "train"
split_map.loc[X_val.index] = "val"
split_map.loc[X_test.index] = "test"

OPTIONAL_CONTEXT_COLS = [
    "tx_count_total",
    "tx_amount_sum",
    "tx_amount_mean",
    "tx_freq_per_day",
    "top_email_domain",
    "identity_present",
    "low_history_flag",
    "history_support_score",
]


def export_top5000_clients(model_name, suffix):
    model = fitted_models[model_name]
    print("Считаю fraud risk-score для всех клиентов через модель:", model_name)

    X_all_enc = preprocessor.transform(X)
    all_scores = model.predict_proba(X_all_enc)[:, 1]

    scored = pd.DataFrame({
        "client_id": df.loc[X.index, "client_id"].values if "client_id" in df.columns else X.index.astype(str),
        "fraud_risk_score": all_scores,
        "profile_fraud_label": y.loc[X.index].values,
        "split": split_map.loc[X.index].values,
        "model_name": model_name,
    }, index=X.index)

    for col in OPTIONAL_CONTEXT_COLS:
        if col in df.columns:
            scored[col] = df.loc[X.index, col].values

    scored = scored.sort_values("fraud_risk_score", ascending=False).reset_index(drop=False)
    scored = scored.rename(columns={"index": "source_row_index"})
    scored.insert(0, "rank", np.arange(1, len(scored) + 1))

    top5000_all = scored.head(5000).copy()
    all_path = RESULTS_DIR / f"ml_v1_4_top5000_clients_{suffix}.csv"
    top5000_all.to_csv(all_path, index=False)

    top5000_test = scored[scored["split"] == "test"].head(5000).copy()
    test_path = RESULTS_DIR / f"ml_v1_4_top5000_test_clients_{suffix}.csv"
    top5000_test.to_csv(test_path, index=False)

    print("Top-5000 по всем клиентам сохранен в:", all_path)
    print("Top-5000 по test сохранен в:", test_path)
    print("Доля fraud в общем Top-5000:", float(top5000_all["profile_fraud_label"].mean()))
    print("Доля fraud в test Top-5000:", float(top5000_test["profile_fraud_label"].mean()) if len(top5000_test) else None)
    display(top5000_all.head(20))
    return top5000_all, top5000_test


top5000_by_pr, top5000_test_by_pr = export_top5000_clients(
    best_by_pr_name,
    "by_pr_auc_model",
)

top5000_by_roc, top5000_test_by_roc = export_top5000_clients(
    best_by_roc_name,
    "by_roc_auc_model",
)

## Как читать результаты v1.4

Основная проверка остается по `PR-AUC` и top-k, потому что наш сценарий - risk-list / triage. Но теперь обязательно смотрим и ROC/FPR-таблицу: `Recall@FPR=1%`, `Recall@FPR=3%`, `Recall@FPR=5%`. Если модель немного хуже по `PR-AUC`, но заметно лучше при малом `FPR`, ее можно рассматривать как ROC/FPR-oriented режим для банка.

Если `v1.4` не переходит через `0.50 PR-AUC`, это не провал. Это значит, что алгоритмический перебор почти исчерпан, и следующим шагом (`v1.5`) лучше делать новый feature engineering: graph/linkage признаки через card/email/device/identity связи, temporal burst features или сглаженные ratio-признаки для клиентов с короткой историей.